In [ ]:
# (DO NOT containerize this cell)
# Data handling parameters
param_minio_endpoint = 'scruffy.lab.uvalight.net:9000'
param_minio_virtual_lab_bucket = 'naa-vre-laserfarm'
param_max_nodes = 2

param_retile_ouput = '_retile_record.js'
param_removed_from_input = '.LAZ'
param_folder_original = 'original'
param_folder_to_retile = 'toRetile'
param_folder_retiled = 'retiled'

In [ ]:
# Secrets (DO NOT containerize this cell)
from SecretsProvider import SecretsProvider
from getpass import getpass

secrets_provider = SecretsProvider(input_func=getpass)
secret_minio_access_key = secrets_provider.get_secret('secret_minio_access_key')
secret_minio_secret_key = secrets_provider.get_secret('secret_minio_secret_key')

In [ ]:
# (DO NOT containerize this cell)
from minio import Minio
from pathlib import Path
from typing import List
import math
import json
import io
import yaml
import random

In [ ]:
# Retiling batching input
folder_batch_from = param_folder_original
batch_accounting_folder = param_folder_to_retile
folder_processed = param_folder_retiled
removed_from_input = param_removed_from_input
replaced_by_in_output = param_retile_ouput

In [ ]:
# Create file batches
def get_unprocessed_filenames(replaced_by_in_output: str, removed_from_input: str, filenames: List[str], batch_accounting_folder: str) -> List[str]:
    objects = minio_client.list_objects(param_minio_virtual_lab_bucket, prefix=folder_processed, recursive=True)
    processed_filenames = {str(Path(obj.object_name).name) for obj in objects}
    processed_files_before_processing = {filename.replace(replaced_by_in_output, removed_from_input) for filename in processed_filenames}
    files_to_process = [filename for filename in filenames if filename not in processed_files_before_processing]
    return files_to_process

def create_workflow_files(batches: int, batch_folder: str, file_folder: str, unprocessed_filenames: List[str]) -> List[str]:
    if not unprocessed_filenames:
        print(f"The list of unprocessed_filenames is empty. Found {len(filenames)} files which have alreay been processed.")
        return []
    else:
        return create_batches(batches, batch_folder, file_folder, unprocessed_filenames)

def create_batches(batches: int, batch_folder: str, file_folder: str, filenames: List[str]) -> List[str]:
    random.shuffle(filenames)
    filename_batches = split_into_batches(batches, filenames)
    batch_filenames = []
    for n, batch in enumerate(filename_batches):
        filepath = f"{batch_folder}/{n}.yaml"
        print(f"storing batch: [{batch[0]} ... n={len(batch)}] as json in {filepath}")
        file_information = {
            'folder': file_folder,
            'files': batch
        }
        json_batch = json.dumps(file_information)
        save_json_in_cloud_storage(filepath, json_batch)
        batch_filenames.append(filepath)
    return batch_filenames

def split_into_batches(max_batches: int, filenames: List[str]):
    batch_size = math.ceil(len(filenames)/max_batches)
    return[filenames[i:i + batch_size] for i in range(0, len(filenames), batch_size)]
    
def save_json_in_cloud_storage(filepath: str, json) -> str:
    json_file = io.BytesIO(json.encode('utf-8'))
    minio_client.put_object(bucket_name=param_minio_virtual_lab_bucket, object_name=str(filepath), data=json_file, length=len(json), content_type='application/json')

minio_client = Minio(
    param_minio_endpoint, 
    access_key=secret_minio_access_key,
    secret_key=secret_minio_secret_key,
    secure=True
)

objects = minio_client.list_objects(param_minio_virtual_lab_bucket, prefix=folder_batch_from, recursive=True)
filenames = [str(Path(obj.object_name).name) for obj in objects]
unprocessed_filenames = get_unprocessed_filenames(removed_from_input, replaced_by_in_output, filenames, batch_accounting_folder)
workflow_files: List[str] = create_workflow_files(param_max_nodes, batch_accounting_folder, folder_batch_from, unprocessed_filenames)

In [ ]:
# (DO NOT containerize this cell)
# Dummy use output
minio_client = Minio(
    param_minio_endpoint, 
    access_key=secret_minio_access_key,
    secret_key=secret_minio_secret_key,
    secure=True
)

file_information = yaml.safe_load(minio_client.get_object(bucket_name=param_minio_virtual_lab_bucket, object_name=workflow_files[0]).read().decode('utf-8'))
print(file_information)